# Interactive Clustering Metrics (Section 12.3)

Five-point dataset from the Q&A exercises.  
Select **Method 1** or **Method 2** to see predicted clusters, per-sample silhouette scores, and all internal & external metrics.

**Dataset**

| Sample | Features | GT Cluster | Method 1 | Method 2 |
|--------|----------|-----------|----------|----------|
| S1 | [0, 1] | C1 | C1 | C1 |
| S2 | [1, 1] | C1 | C1 | C2 |
| S3 | [3, 1] | C2 | C2 | C2 |
| S4 | [5, 1] | C2 | C3 | C3 |
| S5 | [6, 1] | C3 | C3 | C3 |


In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from collections import Counter

# ── Dataset ──────────────────────────────────────────────────────────────────
X = [[0, 1], [1, 1], [3, 1], [5, 1], [6, 1]]
GT = [0, 0, 1, 1, 2]        # ground truth (0-indexed: C1=0, C2=1, C3=2)
M1 = [0, 0, 1, 2, 2]        # Method 1 predicted clusters
M2 = [0, 1, 1, 2, 2]        # Method 2 predicted clusters
NAMES = ['S1', 'S2', 'S3', 'S4', 'S5']
COLORS = ['#2563eb', '#dc2626', '#059669']
CLUSTER_LABELS = ['C1', 'C2', 'C3']


# ── Metric helpers ────────────────────────────────────────────────────────────
def euclidean(a, b):
    return math.sqrt(sum((a[i] - b[i]) ** 2 for i in range(len(a))))


def centroid(pts):
    d = len(pts[0])
    return [sum(p[k] for p in pts) / len(pts) for k in range(d)]


def silhouette_scores(labels):
    """Per-sample silhouette scores. Singleton clusters get score 0."""
    n = len(labels)
    unique = sorted(set(labels))
    scores = []
    for i in range(n):
        my_c = labels[i]
        same = [j for j in range(n) if j != i and labels[j] == my_c]
        if not same:
            scores.append(0.0)
            continue
        a_i = sum(euclidean(X[i], X[j]) for j in same) / len(same)
        b_vals = []
        for c in unique:
            if c == my_c:
                continue
            other = [j for j in range(n) if labels[j] == c]
            if other:
                b_vals.append(sum(euclidean(X[i], X[j]) for j in other) / len(other))
        b_i = min(b_vals) if b_vals else 0.0
        denom = max(a_i, b_i)
        scores.append((b_i - a_i) / denom if denom > 0 else 0.0)
    return scores


def mean_silhouette(labels):
    s = silhouette_scores(labels)
    return sum(s) / len(s)


def davies_bouldin(labels):
    """Davies-Bouldin index (lower is better)."""
    n = len(labels)
    unique = sorted(set(labels))
    k = len(unique)
    cents = {c: centroid([X[i] for i in range(n) if labels[i] == c]) for c in unique}
    s = {
        c: sum(euclidean(X[i], cents[c]) for i in range(n) if labels[i] == c)
           / sum(1 for l in labels if l == c)
        for c in unique
    }
    db_sum = 0.0
    for ci in unique:
        max_r = 0.0
        for cj in unique:
            if ci == cj:
                continue
            d = euclidean(cents[ci], cents[cj])
            if d > 0:
                max_r = max(max_r, (s[ci] + s[cj]) / d)
        db_sum += max_r
    return db_sum / k


def dunn_index(labels):
    """Dunn index (higher is better)."""
    n = len(labels)
    unique = sorted(set(labels))
    min_inter = float('inf')
    for u in range(len(unique)):
        for v in range(u + 1, len(unique)):
            c1, c2 = unique[u], unique[v]
            for i in range(n):
                for j in range(n):
                    if labels[i] == c1 and labels[j] == c2:
                        min_inter = min(min_inter, euclidean(X[i], X[j]))
    max_intra = 0.0
    for c in unique:
        idx = [i for i in range(n) if labels[i] == c]
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                max_intra = max(max_intra, euclidean(X[idx[a]], X[idx[b]]))
    return min_inter / max_intra if max_intra > 0 else float('inf')


def rand_index(gt, pred):
    """Rand index (higher is better, range 0–1)."""
    n = len(gt)
    a = d = 0
    for i in range(n):
        for j in range(i + 1, n):
            sg = (gt[i] == gt[j])
            sp = (pred[i] == pred[j])
            if sg and sp:
                a += 1
            elif not sg and not sp:
                d += 1
    return (a + d) / (n * (n - 1) // 2)


def purity(gt, pred):
    """Cluster purity (higher is better, range 0–1)."""
    n = len(gt)
    total = 0
    for c in set(pred):
        cluster_gt = [gt[i] for i in range(n) if pred[i] == c]
        counts = Counter(cluster_gt)
        total += max(counts.values())
    return total / n


def nmi(gt, pred):
    """Normalized Mutual Information (higher is better, range 0–1)."""
    n = len(gt)
    joint = {}
    for g, p in zip(gt, pred):
        joint[(g, p)] = joint.get((g, p), 0) + 1
    gt_c = Counter(gt)
    pred_c = Counter(pred)
    mi = 0.0
    for (g, p), cnt in joint.items():
        p_gp = cnt / n
        p_g = gt_c[g] / n
        p_p = pred_c[p] / n
        mi += p_gp * math.log(p_gp / (p_g * p_p))
    h_gt = -sum((c / n) * math.log(c / n) for c in gt_c.values())
    h_pred = -sum((c / n) * math.log(c / n) for c in pred_c.values())
    return 2 * mi / (h_gt + h_pred) if (h_gt + h_pred) > 0 else 1.0


# ── Visualisation ─────────────────────────────────────────────────────────────
def draw(method='Method 1'):
    labels = M1 if method == 'Method 1' else M2
    sil = silhouette_scores(labels)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # ── Panel 1: Ground truth ──────────────────────────────────────────────
    for i, (pt, g) in enumerate(zip(X, GT)):
        axes[0].scatter(pt[0], pt[1], c=COLORS[g], s=160, zorder=5, edgecolors='white', linewidths=1.5)
        axes[0].annotate(NAMES[i], xy=(pt[0], pt[1]), xytext=(0, 11),
                         textcoords='offset points', ha='center', fontsize=11)
    axes[0].set_title('Ground Truth Labels', fontsize=13)
    axes[0].set_xlim(-0.8, 7.2)
    axes[0].set_ylim(0.55, 1.8)
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    axes[0].grid(alpha=0.25)
    legend_patches = [mpatches.Patch(color=COLORS[c], label=f'GT {CLUSTER_LABELS[c]}') for c in range(3)]
    axes[0].legend(handles=legend_patches, fontsize=9, loc='upper right')

    # ── Panel 2: Predicted clusters ────────────────────────────────────────
    for i, (pt, l) in enumerate(zip(X, labels)):
        axes[1].scatter(pt[0], pt[1], c=COLORS[l], s=160, zorder=5, edgecolors='white', linewidths=1.5)
        axes[1].annotate(NAMES[i], xy=(pt[0], pt[1]), xytext=(0, 11),
                         textcoords='offset points', ha='center', fontsize=11)
    for c in sorted(set(labels)):
        pts = [X[i] for i in range(len(X)) if labels[i] == c]
        cen = centroid(pts)
        axes[1].scatter(cen[0], cen[1], c=COLORS[c], marker='x', s=260, linewidths=3, zorder=6)
    axes[1].set_title(f'{method} Predicted Clusters', fontsize=13)
    axes[1].set_xlim(-0.8, 7.2)
    axes[1].set_ylim(0.55, 1.8)
    axes[1].set_xlabel('Feature 1')
    axes[1].grid(alpha=0.25)
    pred_patches = [mpatches.Patch(color=COLORS[c], label=CLUSTER_LABELS[c])
                    for c in sorted(set(labels))]
    axes[1].legend(handles=pred_patches, fontsize=9, loc='upper right')

    # ── Panel 3: Per-sample silhouette bar chart ───────────────────────────
    bar_colors = [COLORS[l] for l in labels]
    axes[2].bar(range(5), sil, color=bar_colors, edgecolor='white')
    mean_sil = sum(sil) / len(sil)
    axes[2].axhline(mean_sil, color='#0f172a', linestyle='--', linewidth=1.5,
                    label=f'Mean = {mean_sil:.4f}')
    axes[2].axhline(0, color='#94a3b8', linestyle='-', linewidth=0.8)
    axes[2].set_xticks(range(5), NAMES)
    axes[2].set_ylim(-1.05, 1.05)
    axes[2].set_title(f'Per-Sample Silhouette ({method})', fontsize=13)
    axes[2].set_xlabel('Sample')
    axes[2].set_ylabel('Silhouette Score')
    axes[2].legend(fontsize=9)
    axes[2].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

    # ── Metrics table ──────────────────────────────────────────────────────
    sil_val  = mean_silhouette(labels)
    db_val   = davies_bouldin(labels)
    dunn_val = dunn_index(labels)
    ri_val   = rand_index(GT, labels)
    pur_val  = purity(GT, labels)
    nmi_val  = nmi(GT, labels)

    print(f'\n{"="*60}')
    print(f'  Clustering Metrics ── {method}')
    print(f'{"="*60}')
    print(f'  Internal Metrics  (no ground truth required)')
    print(f'  {"─"*56}')
    print(f'  Silhouette Score   : {sil_val:8.4f}   (↑ higher is better, −1 to 1)')
    print(f'  Davies–Bouldin Idx : {db_val:8.4f}   (↓ lower is better,  ≥ 0)')
    print(f'  Dunn Index         : {dunn_val:8.4f}   (↑ higher is better,  > 0)')
    print(f'  {"─"*56}')
    print(f'  External Metrics  (ground truth required)')
    print(f'  {"─"*56}')
    print(f'  Rand Index         : {ri_val:8.4f}   (↑ higher is better,  0 to 1)')
    print(f'  Purity             : {pur_val:8.4f}   (↑ higher is better,  0 to 1)')
    print(f'  NMI                : {nmi_val:8.4f}   (↑ higher is better,  0 to 1)')
    print(f'{"="*60}')


widgets.interact(
    draw,
    method=widgets.Dropdown(
        options=['Method 1', 'Method 2'],
        value='Method 1',
        description='Method:',
    ),
)
